# Cost Decomposition Analysis

Decompose route costs into additive components:
- **cost_calm**: Hull resistance (depends on speed through water)
- **cost_waves**: Wave added resistance (depends on speed through water and wave height)
- **cost_wind**: Wind resistance (depends on speed through wind)

These sum exactly: `cost_total = cost_calm + cost_waves + cost_wind`

Current effects are isolated by comparing kinematics with vs without currents.

We check whether the decomposition fractions are consistent between the single best route
per test case (lowest cost across all runs and both elites) and the top 5% of routes per
test case (month × speed × direction).

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import warnings

from load_tuning_results import add_derived_features, filter_suspicious_routes

warnings.filterwarnings("ignore")

## Load and join data

In [ ]:
# Load decomposed costs (13 per-file parquets)
files = sorted(Path("../results").glob("decomposed_costs_*.parquet"))
df_costs = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(f"Decomposed cost rows: {len(df_costs)} from {len(files)} files")

# Load prelim metadata
gdf = gpd.read_parquet("../results/results_prelim.geoparquet")
gdf = add_derived_features(gdf)
gdf = filter_suspicious_routes(gdf)
gdf = gdf[gdf.forcing_scenario_name == "baseline"].copy()
gdf = gdf.reset_index()

# Join
gdf = gdf.merge(df_costs, on=["filename", "n_elite"], how="inner")
print(f"Joined rows: {len(gdf)}")

# Derived columns
gdf["month"] = pd.to_datetime(gdf.journey_time_start).dt.month_name().str[:3]
gdf["direction"] = gdf.journey_name.map(
    {"Atlantic_forward": "eastward", "Atlantic_backward": "westward"}
)

print(f"Speeds: {sorted(gdf.journey_speed_knots.unique())}")
print(f"Directions: {gdf.direction.unique().tolist()}")
print(f"n_elite values: {sorted(gdf.n_elite.unique())}")

In [ ]:
# Verify decomposition sums exactly
sum_check = gdf.cost_calm + gdf.cost_waves + gdf.cost_wind
max_error = (sum_check - gdf.cost_total).abs().max()
print(f"Decomposition verification: max error = {max_error:.2e}")

# Hazard diagnosis
print(f"\nHazardous routes: {gdf.is_hazardous.sum()}/{len(gdf)}")
print(f"Max wave heights: {gdf.max_wave_height_m.min():.1f} - {gdf.max_wave_height_m.max():.1f} m")

## Define subsets: best route vs top 5% per test case

In [ ]:
# Rank all routes by cost within each test case (month x speed x direction)
gdf["cost_quantile"] = gdf.groupby(
    ["month", "journey_speed_knots", "direction"]
)["elite_cost_absolute"].rank(pct=True)

# Best route per test case (lowest cost across all runs and both elites)
gdf_best = gdf.loc[
    gdf.groupby(["month", "journey_speed_knots", "direction"])["elite_cost_absolute"].idxmin()
].copy()
print(f"Best route per test case: {len(gdf_best)}")

# Top 5% by cost within each test case
gdf_top5 = gdf[gdf.cost_quantile <= 0.05].copy()
print(f"Top 5% routes: {len(gdf_top5)}")

## Component Fractions by Speed and Direction

In [ ]:
def component_fractions(df, label=""):
    fractions = df.groupby(["direction", "journey_speed_knots"]).agg(
        {"cost_calm": "mean", "cost_waves": "mean", "cost_wind": "mean", "cost_total": "mean"}
    )
    for col in ["cost_calm", "cost_waves", "cost_wind"]:
        fractions[f"{col}_pct"] = fractions[col] / fractions["cost_total"] * 100
    if label:
        print(f"\n{label}")
    return fractions[["cost_calm_pct", "cost_waves_pct", "cost_wind_pct"]].round(1)

display(component_fractions(gdf_best, "Best route per test case"))
display(component_fractions(gdf_top5, "Top 5% per test case"))

## Cost Components by Month, Speed, and Direction

Comparing best route per test case (solid bars) with top 5% (faint bars).

In [ ]:
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
components = ["cost_calm", "cost_waves", "cost_wind"]
colors = ["tab:blue", "tab:orange", "tab:green"]

fig, axes = plt.subplots(2, 3, sharey=True)

for row, direction in enumerate(["eastward", "westward"]):
    for col, speed in enumerate([8, 10, 12]):
        ax = axes[row, col]
        w = 0.35

        for subset, df_sub, offset, alpha in [
            ("best", gdf_best, -w/2, 1.0),
            ("top5%", gdf_top5, w/2, 0.5),
        ]:
            sel = df_sub[(df_sub.journey_speed_knots == speed) & (df_sub.direction == direction)]
            agg = sel.groupby("month")[components].mean().reindex(month_order) / 1e12

            bottom = agg[components[0]] * 0
            for i, comp in enumerate(components):
                label = comp.replace("cost_", "") if (row == 0 and col == 0 and subset == "best") else None
                ax.bar([xi + offset for xi in range(12)], agg[comp], bottom=bottom,
                       color=colors[i], alpha=alpha, label=label, width=w)
                bottom += agg[comp]

        # Mark hazardous months
        sel_all = gdf[(gdf.journey_speed_knots == speed) & (gdf.direction == direction)]
        hazard_by_month = sel_all.groupby("month").is_hazardous.any().reindex(month_order)
        for i, (month, is_haz) in enumerate(hazard_by_month.items()):
            if is_haz:
                ax.axvspan(i - 0.5, i + 0.5, color="red", alpha=0.1, zorder=0)

        ax.set_xticks(range(12))
        ax.set_xticklabels(month_order, rotation=90)
        if row == 0:
            ax.set_title(f"{speed} kn")
        ax.set_xlabel("")
        if col == 0:
            ax.set_ylabel(f"{direction}\nCost (TJ)")

axes[0, 0].legend(fontsize=8)
fig.text(0.99, 0.01, "left bar=best route, right bar=top 5%; red=hazardous", ha="right", fontsize=7)
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig("../figures/022_cost_components_stacked.png")
plt.savefig("../figures/022_cost_components_stacked.pdf")

## Current Effects by Direction

In [ ]:
def current_effect_table(df, label=""):
    effects = df.groupby(["direction", "journey_speed_knots"]).agg(
        {"delta_current_on_calm": "mean", "delta_current_on_waves": "mean",
         "delta_current_total": "mean", "cost_calm": "mean", "cost_total": "mean"}
    )
    effects["delta_calm_pct"] = effects["delta_current_on_calm"] / effects["cost_calm"] * 100
    effects["delta_total_pct"] = effects["delta_current_total"] / effects["cost_total"] * 100
    if label:
        print(f"\n{label}")
    return effects[["delta_calm_pct", "delta_total_pct"]].round(1)

display(current_effect_table(gdf_best, "Best route per test case"))
display(current_effect_table(gdf_top5, "Top 5% per test case"))

In [ ]:
fig, axes = plt.subplots(2, 3, sharey="row")

for row, direction in enumerate(["eastward", "westward"]):
    for col, speed in enumerate([8, 10, 12]):
        ax = axes[row, col]

        for subset, df_sub, alpha in [("best", gdf_best, 1.0), ("top5%", gdf_top5, 0.35)]:
            sel = df_sub[(df_sub.journey_speed_knots == speed) & (df_sub.direction == direction)]
            agg = (
                sel.groupby("month")[["delta_current_on_calm", "delta_current_on_waves"]]
                .mean().reindex(month_order) / 1e12
            )
            x = range(12)
            w = 0.35
            offset = -w/2 if subset == "best" else w/2
            ax.bar([xi + offset for xi in x], agg["delta_current_on_calm"],
                   width=w, alpha=alpha, color="tab:blue",
                   label="calm" if (row == 0 and col == 0 and subset == "best") else None)
            ax.bar([xi + offset for xi in x], agg["delta_current_on_waves"],
                   width=w, alpha=alpha, color="tab:orange",
                   bottom=agg["delta_current_on_calm"],
                   label="waves" if (row == 0 and col == 0 and subset == "best") else None)

        ax.axhline(0, color="gray", linestyle="--")

        # Mark hazardous months
        sel_all = gdf[(gdf.journey_speed_knots == speed) & (gdf.direction == direction)]
        hazard_by_month = sel_all.groupby("month").is_hazardous.any().reindex(month_order)
        for i, (month, is_haz) in enumerate(hazard_by_month.items()):
            if is_haz:
                ax.axvspan(i - 0.5, i + 0.5, color="red", alpha=0.1, zorder=0)

        ax.set_xticks(range(12))
        ax.set_xticklabels(month_order, rotation=90)
        if row == 0:
            ax.set_title(f"{speed} kn")
        ax.set_xlabel("")
        if col == 0:
            ax.set_ylabel(f"{direction}\nCurrent effect (TJ)")

axes[0, 0].legend(fontsize=8)
fig.text(0.99, 0.01, "left bar=best route, right bar=top 5%; red=hazardous", ha="right", fontsize=7)
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig("../figures/022_current_effects.png")
plt.savefig("../figures/022_current_effects.pdf")

## Consistency check: best elite vs top 5%

In [ ]:
# Compare component fractions between best route and top 5%
frac_best = component_fractions(gdf_best)
frac_top5 = component_fractions(gdf_top5)
diff = (frac_best - frac_top5).abs()

print("Absolute difference in component fractions (pp):")
display(diff.round(2))

print(f"\nMax difference: {diff.max().max():.2f} pp")
print(f"Mean difference: {diff.mean().mean():.2f} pp")

## Summary

In [ ]:
# Summary table for best routes per test case
summary_rows = []
for direction in ["eastward", "westward"]:
    for speed in [8, 10, 12]:
        df_sub = gdf_best[
            (gdf_best.journey_speed_knots == speed) & (gdf_best.direction == direction)
        ]
        df_agg = df_sub.groupby("month")[
            ["cost_calm", "delta_current_on_calm", "is_hazardous", "max_wave_height_m"]
        ].agg({
            "cost_calm": "mean",
            "delta_current_on_calm": "mean",
            "is_hazardous": "any",
            "max_wave_height_m": "max",
        }).reindex(month_order)
        df_agg["pct"] = df_agg["delta_current_on_calm"] / df_agg["cost_calm"] * 100
        df_agg["speed"] = speed
        df_agg["direction"] = direction
        summary_rows.append(df_agg.reset_index())

summary_df = pd.concat(summary_rows)
summary_df.to_csv("../results/022_cost_decomposition_summary.csv", index=False)
print("Saved to ../results/022_cost_decomposition_summary.csv")

# Summary of non-hazardous routes
gdf_safe = gdf_best[~gdf_best.is_hazardous]
print(f"\n=== Non-hazardous routes: {len(gdf_safe)}/{len(gdf_best)} ===")
if len(gdf_safe) > 0:
    safe_effects = gdf_safe.groupby("direction").agg(
        {"delta_current_total": "mean", "cost_total": "mean"}
    )
    safe_effects["pct"] = (
        safe_effects["delta_current_total"] / safe_effects["cost_total"] * 100
    )
    print(safe_effects[["pct"]].round(1))